In [1]:
import os

In [2]:
%pwd

'e:\\End-to-End-Chest-Disease-Classification-Using-Machine-Learning\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\End-to-End-Chest-Disease-Classification-Using-Machine-Learning'

In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    train_data: Path
    valid_data: Path
    params_epoch: int
    params_batch_size: int
    params_image_size: list
    params_is_augmentation: bool

In [11]:
from ChestDiseaseClassification.constant import *
from ChestDiseaseClassification.utils.common import read_yaml, create_directories
import tensorflow as tf


In [12]:
class ConfiguartionManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        base_path = os.path.join(self.config.data_ingestion.unzip_dir, "Chest-CT-Scan-Data")
        train_data = os.path.join(base_path, "train")
        valid_data = os.path.join(base_path, "valid")
        create_directories([training.root_dir])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),

            train_data=Path(train_data),
            valid_data=Path(valid_data),
            params_epoch=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_image_size=params.IMAGE_SIZE,
            params_is_augmentation=params.AUGMENTATION,
        )

        return training_config

In [13]:

import tensorflow as tf


In [16]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)
        self.model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"]
    )
    
    def train_valid_generator(self):
        datagenerator_kwargs = dict(rescale=1./255)
        dataflow_kwargs = dict(target_size=self.config.params_image_size[:-1], batch_size=self.config.params_batch_size, interpolation="bilinear",class_mode="categorical")
        
         # Training Generator
        train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            horizontal_flip=True,
            width_shift_range=0.2,
            height_shift_range=0.2,
            rotation_range=40,
            zoom_range=0.2,
            shear_range=0.2,
            **datagenerator_kwargs
        )

         # Validation Generator (NO augmentation)
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)


        self.train_generator = train_datagenerator.flow_from_directory(
                directory=self.config.train_data,
                shuffle=True,
                **dataflow_kwargs
            )
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.valid_data,
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)


    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epoch,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps
        )

        self.save_model(path=self.config.trained_model_path, model=self.model)


In [17]:
try:
    config = ConfiguartionManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()

except Exception as e:
    raise e

[2026-02-24 10:35:12,313:INFO:common:yaml file: config\config.yaml loaded successfully]
[2026-02-24 10:35:12,316:INFO:common:yaml file: params.yaml loaded successfully]
[2026-02-24 10:35:12,318:INFO:common:directory created at: artifacts]
[2026-02-24 10:35:12,320:INFO:common:directory created at: artifacts/training]
[2026-02-24 10:35:12,496:WARNING:saving_utils:Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 613 images belonging to 4 classes.
Found 72 images belonging to 4 classes.
38/38 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.4456 - loss: 1.5694 - val_accuracy: 0.5156 - val_loss: 1.6076
[2026-02-24 10:36:33,022:WARNING:saving_api:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.sa